# GEPA: оптимизация промпта для русской орфографии

Вместо SFT (дообучение весов) здесь используется [GEPA](https://github.com/gepa-ai/gepa) (Genetic-Pareto) — эволюционный оптимизатор **промптов**. Базовая модель **Qwen3.5-0.8B** остаётся замороженной; GEPA ищет лучшую system-инструкцию на датасете [`BW/RU_SPELLCHECK_DEVICE`](https://huggingface.co/datasets/BW/RU_SPELLCHECK_DEVICE).

**Нужно:** GPU (Colab T4+) для Qwen, `API_KEY` для reflection LM ([Cloud.ru Foundation Models](https://foundation-models.api.cloud.ru/v1)).

**Runtime → Run all**

## 1. Авторизация и API-ключи

In [1]:
import os
from huggingface_hub import login

API_BASE_URL = "https://foundation-models.api.cloud.ru/v1"
REFLECTION_MODEL = "openai/gpt-oss-120b"

# Colab: можно использовать секреты
# from google.colab import userdata
# hf_token = userdata.get('HF_TOKEN')
# api_key = userdata.get('API_KEY')

hf_token = 'fake'#os.environ.get('HF_TOKEN', '')
api_key = 'fake'

if hf_token:
    login(hf_token)
if not api_key:
    print('⚠️  API_KEY не задан — reflection LM не сможет работать')


## 2. Установка зависимостей

In [2]:
%%capture
import os, re

!pip install gepa openai

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec

import torch
torch._dynamo.config.recompile_limit = 64


## 3. Загрузка базовой модели Qwen3.5-0.8B

In [3]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

MODEL_NAME = "unsloth/Qwen3.5-0.8B"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    full_finetuning=False,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-instruct",
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

## 4. Подготовка данных для GEPA

In [4]:
from datasets import load_dataset
from gepa.adapters.default_adapter.default_adapter import DefaultDataInst

DATASET_NAME = "BW/RU_SPELLCHECK_DEVICE"
dataset = load_dataset(DATASET_NAME)


def to_gepa_examples(split) -> list[DefaultDataInst]:
    examples = []
    for row in split:
        examples.append({
            "input": row["typed"],
            "answer": row["original"],
            "additional_context": {
                "device_type": row.get("device_type", "unknown"),
            },
        })
    return examples


trainset = to_gepa_examples(dataset["train"])
valset = to_gepa_examples(dataset["test"])
testset = valset  # официальный test split

print(f"Train: {len(trainset)}, Val/Test: {len(valset)}")
print(f"Пример input:  {trainset[0]['input']}")
print(f"Пример answer: {trainset[0]['answer']}")


Train: 296, Val/Test: 296
Пример input:  Две молодые демушки вели ее под рукию
Пример answer: Две молодые девушки вели ее под руки.


## 5. Метрики и evaluator для GEPA

In [5]:
import difflib
import re

from gepa.adapters.default_adapter.default_adapter import EvaluationResult


def tokenize_spellcheck(text: str) -> list[str]:
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def extract_edits(source: str, target: str) -> dict[tuple[int, int], tuple[str, ...]]:
    src_tokens = tokenize_spellcheck(source)
    tgt_tokens = tokenize_spellcheck(target)
    edits: dict[tuple[int, int], tuple[str, ...]] = {}
    matcher = difflib.SequenceMatcher(None, src_tokens, tgt_tokens)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        replacement = tuple(tgt_tokens[j1:j2])
        if src_tokens[i1:i2] != list(replacement):
            edits[(i1, i2)] = replacement
    return edits


def edit_f1(source: str, correction: str, prediction: str) -> dict[str, float]:
    gold = extract_edits(source, correction)
    pred = extract_edits(source, prediction)
    tp = sum(1 for key, val in pred.items() if gold.get(key) == val)
    n_pred, n_gold = len(pred), len(gold)
    precision = tp / n_pred if n_pred else 1.0
    recall = tp / n_gold if n_gold else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "num_gold": n_gold,
        "num_pred": n_pred,
    }


def format_edits(edits: dict[tuple[int, int], tuple[str, ...]], src_tokens: list[str]) -> str:
    if not edits:
        return "(нет правок)"
    lines = []
    for (i, j), repl in sorted(edits.items()):
        src_part = " ".join(src_tokens[i:j])
        tgt_part = " ".join(repl)
        lines.append(f"  '{src_part}' → '{tgt_part}'")
    return "\n".join(lines)


class SpellcheckEditF1Evaluator:
    """Edit-level F1 как score + подробный feedback для reflection LM."""

    def __call__(self, data, response: str) -> EvaluationResult:
        source = data["input"]
        correction = data["answer"]
        prediction = response.strip()

        metrics = edit_f1(source, correction, prediction)
        score = metrics["f1"]

        src_tokens = tokenize_spellcheck(source)
        gold_edits = extract_edits(source, correction)
        pred_edits = extract_edits(source, prediction)

        missed = {k: v for k, v in gold_edits.items() if pred_edits.get(k) != v}
        extra = {k: v for k, v in pred_edits.items() if gold_edits.get(k) != v}

        feedback_parts = [
            f"Исходный текст: {source}",
            f"Эталон: {correction}",
            f"Предсказание: {prediction}",
            f"Edit F1: {metrics['f1']:.3f} (P={metrics['precision']:.3f}, R={metrics['recall']:.3f})",
            f"Устройство ввода: {data['additional_context'].get('device_type', 'unknown')}",
        ]

        if prediction == correction:
            feedback_parts.append("✓ Точное совпадение с эталоном.")
        else:
            if missed:
                feedback_parts.append("Пропущенные правки (нужно исправить, но модель не исправила):")
                feedback_parts.append(format_edits(missed, src_tokens))
            if extra:
                feedback_parts.append("Лишние/неверные правки (модель изменила лишнее):")
                feedback_parts.append(format_edits(extra, src_tokens))
            if len(prediction) > len(correction) * 1.3:
                feedback_parts.append("Проблема: ответ слишком длинный — возможно, модель перефразирует вместо минимальной правки.")

        return EvaluationResult(score=score, feedback="\n".join(feedback_parts))


## 6. Обёртка Qwen для GEPA DefaultAdapter

In [7]:
class QwenChatModel:
    """ChatCompletionCallable для локальной Qwen через Unsloth."""

    def __init__(self, model, tokenizer, max_new_tokens: int = 256):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens

    def __call__(self, messages: list[dict]) -> str:
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = self.tokenizer(
            text=[text],
            images=None,
            videos=None,
            return_tensors="pt",
        ).to(self.model.device)

        input_len = inputs["input_ids"].shape[1]
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
        )
        return self.tokenizer.decode(
            outputs[0][input_len:],
            skip_special_tokens=True,
        ).strip()


task_lm = QwenChatModel(model, tokenizer)
evaluator = SpellcheckEditF1Evaluator()


## 7. Запуск GEPA

In [8]:
import gepa
from openai import OpenAI

SEED_PROMPT = (
    "Ты корректор русского текста. Исправь орфографические, пунктуационные "
    "и регистровые ошибки в предложении пользователя. "
    "Верни только исправленный текст без пояснений и кавычек."
)

seed_candidate = {"system_prompt": SEED_PROMPT}
# valset = valset[:30]     # уменьшить val
# trainset = trainset[:50]
REFLECTION_LM = REFLECTION_MODEL
reflection_client = OpenAI(api_key=api_key, base_url=API_BASE_URL)


def reflection_lm(prompt: str | list[dict]) -> str:
    """Cloud.ru через OpenAI SDK (как в test_api.py; litellm даёт 404 на этом endpoint)."""
    messages = (
        [{"role": "user", "content": prompt}]
        if isinstance(prompt, str)
        else prompt
    )
    response = reflection_client.chat.completions.create(
        model=REFLECTION_MODEL,
        messages=messages,
        max_tokens=2500,
        temperature=0.5,
        presence_penalty=0,
        top_p=0.95,
    )
    return response.choices[0].message.content

MAX_METRIC_CALLS = 2000  # бюджет оценок; увеличьте для лучшего результата
RUN_DIR = "gepa_spellcheck_run"  # перед новым запуском удалите папку, иначе GEPA продолжит старый run

result = gepa.optimize(
    seed_candidate=seed_candidate,
    trainset=trainset,
    valset=valset,
    task_lm=task_lm,
    evaluator=evaluator,
    reflection_lm=reflection_lm,
    max_metric_calls=MAX_METRIC_CALLS,
    reflection_minibatch_size=5,
    run_dir=RUN_DIR,
    display_progress_bar=True,
    seed=3407,
)

print("\n=== Лучший промпт ===")
print(result.best_candidate["system_prompt"])
print(f"\nЛучший val score (avg): {result.val_aggregate_scores[result.best_idx]:.4f}")
print(f"Всего кандидатов: {result.num_candidates}, metric calls: {result.total_metric_calls}")


GEPA Optimization:   0%|          | 0/2000 [00:00<?, ?rollouts/s]Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not

Iteration 0: Base program full valset score: 0.003003003003003003 over 296 / 296 examples
Iteration 1: Selected program 0 score: 0.003003003003003003


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Iteration 1: Proposed new text for system_prompt: You are a Russian language proofreader.  
Your job is to correct **only** orthographic (spelling), punctuation, and case (capitalization) errors in the user‑provided Russian text.  

**How to work**

1. **Read the entire input text** (it may contain one or several sentences).  
2. **Identify and fix**:
   - Misspelled words, including letters swapped, missing, or extra.  
   - Incorrect capitalization (the first word of a sentence, proper nouns, etc.).  
   - Missing, extra, or misplaced punctuation marks (commas, periods, exclamation points, question marks, ellipsis, quotation marks, dash, etc.).  
   - Incorrect word forms that affect spelling (e.g., “бытьть” → “быть”, “прокралывался” → “прокрадывался”).  
3. **Do not change** the wording or style beyond what is required for correct Russian spelling, punctuation, and case. Do not add, delete, or rephrase sentences unless a change is strictly necessary to fix an error.  
4. **Output** 

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


...

Iteration 22: Proposed new text for system_prompt: You are a Russian‑language proofreader.  
Your ONLY task is to correct **orthographic (spelling), punctuation, and capitalization** mistakes in the text that the user provides.  
Everything else must stay exactly as it was: wording, word order, line breaks, spaces that are not part of an error, numbers, emojis, other symbols, and any hyphens that belong to a word (e.g., «по‑русски», dates like «2024‑05‑01», etc.) must be preserved.

### What you may change
1. **Spelling** – fix missing, extra, swapped, or wrong letters; replace a misspelled word with the correct one (e.g., `инжерер → инженер`, `тто → что`). Do **not** add or delete whole words unless the change is required to turn a misspelled word into the correct one.
2. **Capitalization** –  
   * First word of a sentence → capital letter.  
   * Proper names, geographical names, titles, etc. → first letter capital, the rest lower‑case (unless the word is an established abbreviation

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Iteration 22: New subsample score 0.4 is better than old score 0.3333333333333333. Continue to full eval and add to candidate pool.


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Iteration 22: Valset score for new program: 0.07295824795824796 (coverage 296 / 296)
Iteration 22: Val aggregate for new program: 0.07295824795824796
Iteration 22: Individual valset scores for new program: {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.2857142857142857, 4: 0.0, 5: 0.6666666666666666, 6: 0.0, 7: 0.0, 8: 0.4, 9: 0.0, 10: 0.0, 11: 0.0, 12: 0.0, 13: 0.4, 14: 0.0, 15: 0.0, 16: 0.0, 17: 0.6666666666666666, 18: 0.0, 19: 0.25, 20: 0.1818181818181818, 21: 0.0, 22: 0.3333333333333333, 23: 0.0, 24: 0.0, 25: 0.0, 26: 0.0, 27: 0.0, 28: 0.0, 29: 0.0, 30: 0.0, 31: 0.5, 32: 0.0, 33: 0.0, 34: 0.22222222222222224, 35: 0.0, 36: 0.0, 37: 0.0, 38: 0.3333333333333333, 39: 0.0, 40: 0.0, 41: 0.0, 42: 0.28571428571428575, 43: 0.0, 44: 0.0, 45: 0.0, 46: 0.0, 47: 0.0, 48: 0.0, 49: 0.0, 50: 0.0, 51: 0.0, 52: 0.0, 53: 0.0, 54: 0.0, 55: 0.0, 56: 0.0, 57: 0.0, 58: 0.0, 59: 0.0, 60: 0.0, 61: 0.0, 62: 0.0, 63: 0.2222222222222222, 64: 0.0, 65: 0.0, 66: 0.0, 67: 0.0, 68: 0.0, 69: 0.0, 70: 0.0, 71: 0.0, 72: 0.0, 73: 0.0,

## 8. Сохранение результата

In [10]:
import json
from pathlib import Path

from gepa.adapters.default_adapter.default_adapter import DefaultAdapter


def evaluate_prompt_on_testset(prompt: str, examples: list) -> dict:
    adapter = DefaultAdapter(model=task_lm, evaluator=evaluator)
    candidate = {"system_prompt": prompt}
    batch = adapter.evaluate(examples, candidate, capture_traces=False)

    per_example = []
    exact = 0
    for ex, out, score in zip(examples, batch.outputs, batch.scores):
        prediction = out["full_assistant_response"].strip()
        gold = ex["answer"].strip()
        is_exact = prediction == gold
        exact += int(is_exact)
        per_example.append({
            "input": ex["input"],
            "gold": ex["answer"],
            "prediction": prediction,
            "edit_f1": round(score, 4),
            "exact_match": is_exact,
            "device_type": ex["additional_context"].get("device_type", "unknown"),
        })

    total = len(examples)
    return {
        "aggregate": {
            "avg_edit_f1": round(sum(batch.scores) / total, 4),
            "exact_match": round(exact / total, 4),
            "exact_match_count": exact,
            "total": total,
        },
        "examples": per_example,
    }


print("Оценка на test до GEPA (seed prompt)...")
test_before_gepa = evaluate_prompt_on_testset(SEED_PROMPT, testset)

print("Оценка на test после GEPA (optimized prompt)...")
test_after_gepa = evaluate_prompt_on_testset(result.best_candidate["system_prompt"], testset)

OUTPUT_PATH = Path("gepa_spellcheck_prompt.json")
payload = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "test_split": "test",
    "seed_prompt": SEED_PROMPT,
    "optimized_prompt": result.best_candidate["system_prompt"],
    "val_scores": result.val_aggregate_scores,
    "max_metric_calls": MAX_METRIC_CALLS,
    "reflection_lm": REFLECTION_LM,
    "reflection_api_base": API_BASE_URL,
    "test_evaluation": {
        "before_gepa": test_before_gepa,
        "after_gepa": test_after_gepa,
        "delta": {
            "avg_edit_f1": round(
                test_after_gepa["aggregate"]["avg_edit_f1"]
                - test_before_gepa["aggregate"]["avg_edit_f1"],
                4,
            ),
            "exact_match": round(
                test_after_gepa["aggregate"]["exact_match"]
                - test_before_gepa["aggregate"]["exact_match"],
                4,
            ),
        },
    },
}
OUTPUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Сохранено: {OUTPUT_PATH.resolve()}")
print("Test до GEPA:", test_before_gepa["aggregate"])
print("Test после GEPA:", test_after_gepa["aggregate"])


Сохранено: /content/gepa_spellcheck_prompt.json


## 9. Оценка seed vs optimized на test

In [ ]:
print("Seed prompt (test):")
print(test_before_gepa["aggregate"])
print("\nOptimized prompt (test):")
print(test_after_gepa["aggregate"])
print("\nDelta:")
print(payload["test_evaluation"]["delta"])


## 10. Примеры предсказаний

In [ ]:
adapter = DefaultAdapter(model=task_lm, evaluator=evaluator)
optimized_candidate = {"system_prompt": result.best_candidate["system_prompt"]}
demo_batch = adapter.evaluate(testset[:5], optimized_candidate, capture_traces=False)

for ex, out, score in zip(testset[:5], demo_batch.outputs, demo_batch.scores):
    print("---")
    print(f"typed:    {ex['input']}")
    print(f"gold:     {ex['answer']}")
    print(f"pred:     {out['full_assistant_response']}")
    print(f"edit F1:  {score:.3f}")
